# Deployment 5 — state-gated flipbook (autos)

Flip through per-file autocorr data gated on RF-switch state, using
`eigsep_data.MetadataIndex`: one row per integration for every readable
filtered file, scanned once and then served from a sidecar cache next to
the data. `StateBrowser` (`eigsep_data.browse`) reads only the selected
rows of the file on screen.

State breakdown over the 5120 readable filtered files (1.23M integrations),
measured from the index:

| state | files | integrations | what |
|---|---|---|---|
| `RFANT` | 3243 | 733k (59.7%) | antenna |
| `MISSING` | 2470 | 465k (37.9%) | no switch metadata reached the writer |
| `RFAMB` | 134 | 13.2k (1.1%) | ambient load |
| `RFNON` | 101 | 7.6k (0.6%) | noise source |
| `RFSP1` | 20 | 2.8k | spare-1 port (termination in `potmon_sp1_term_name`) |
| `VNA*` (9 states) | 19–38 each | 0.4k–0.9k each | VNA cal states |
| `UNKNOWN` | 198 | 822 | producer-flagged transition/error rows |
| `RFNOFF` | 3 | 70 | noise source off |

Caveats:
- The index keeps two kinds of "no state" apart: `UNKNOWN` is the *producer* saying the integration is contaminated (switch transition, error status), `MISSING` means no metadata reached the writer at all. The old `rfswitch_index.npz` lumped both into `UNKNOWN`.
- **1842 files have no rfswitch metadata at all** (most of Jul 15) — every one of their rows is `MISSING`, so `rfswitch="RFANT"` silently excludes them. Browse `rfswitch="MISSING"` to eyeball those (2470 files hold at least one `MISSING` row).
- The state vocabulary is open: 16 distinct values here, and a deployment can add more. Selecting a state the data does not have gives an empty selection, not a crash.
- Cal states cluster on **Jul 17** (the cal cycle: ~84 ant → ~110 noise → ~42 load rows per file), with scattered visits Jul 13/14/16/18.
- Switch transitions are buffered by `UNKNOWN` rows, so gated rows are clean.

Controls: Play button auto-flips; slider scrubs. Top panel = waterfall of the file's gated rows (first key the file actually carries), bottom = mean spectrum per key with 10–90% envelope.

In [ ]:
%matplotlib widget
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from eigsep_data import MetadataIndex
from eigsep_data.browse import StateBrowser

# repo root = first parent dir containing the filtered data
REPO = Path.cwd().resolve()
while not (REPO / "data" / "deployment5_filtered").is_dir():
    if REPO.parent == REPO:
        raise FileNotFoundError("data/deployment5_filtered not found above cwd")
    REPO = REPO.parent
DATA_DIR = REPO / "data" / "deployment5_filtered"

# A cold scan of the deployment the first time, cached after:
# ~65 s for the 5120 files / 1.23M rows, then ~5 s from the sidecar.
idx = MetadataIndex(DATA_DIR)

In [ ]:
b = StateBrowser(idx.select(rfswitch="RFANT"), keys=["0", "4"])
# phases A/B have other live keys, e.g.:
#   StateBrowser(idx.select(rfswitch="RFANT", filter_phase="B"),
#                keys=["3", "4"])

In [ ]:
# load-only / noise-only / no-metadata browsing
# (own figure each; close old ones when done)
# b_load = StateBrowser(idx.select(rfswitch="RFAMB"), keys=["0", "4"])
# b_noise = StateBrowser(idx.select(rfswitch="RFNON"), keys=["0", "4"])
# b_gap = StateBrowser(idx.select(rfswitch="MISSING"), keys=["0", "4"])

## Scratch

In [ ]:
plt.close("all")  # run when widget figures pile up -- each holds its arrays